In [1]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift
from utility.my_colors import pink

from sparcl.client import SparclClient
from dl import queryClient as qc

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
print(qc.schema('desi_dr1.agngal'))


Schema: desi_dr1
 Table: agngal

     Column Name   Description
     -----------   -----------
               z   Redshift from the FastSpecFit catalog
            zerr   Redshift error
       target_ra   Right Ascension in decimal degrees (J2000)
      target_dec   Declination in decimal degrees (J2000)
         min_mjd   MJD of the first exposure in coadded spectrum
        mean_mjd   Mean of MJD of exposures in a coadded spectrum
         max_mjd   MJD of the last exposure in coadded spectrum
            elon   Ecliptic longitude
            elat   Ecliptic latitude
            glon   Galactic longitude
            glat   Galactic latitude
        targetid   Unique identifier for each object observed by DESI
           zwarn   Warning flags (0 is good)
           ls_id   Legacy Surveys DR9 identifier from RELEASE, OBJID, BRICKID
     desi_target   DESI (dark time program) target selection bitmask
     scnd_target   SCND (secondary program) target selection bitmask
      bgs_target 

In [ ]:
# query = """
#         SELECT targetid, z, zerr, zwarn, civ_1549_flux, civ_1549_flux_ivar, civ_1549_sigma, flux_w3, flux_ivar_w3, spectype, survey
#         FROM desi_dr1.agngal
#         WHERE (z BETWEEN 2 AND 3.4) AND (zwarn=0) AND (civ_1549_flux>0) AND (spectype='QSO')
#         """

query = """
        SELECT targetid, z, zerr, zwarn, civ_1549_flux, civ_1549_flux_ivar, civ_1549_sigma, flux_w3, flux_ivar_w3, spectype, survey
        FROM desi_dr1.agngal
        WHERE (zwarn=0) AND (spectype='QSO') AND (z BETWEEN 2.2 AND 4.38) AND (civ_1549_flux > 0)
        """

In [11]:
results = qc.query(sql=query, fmt="pandas")

In [ ]:
results[results["targetid"]==39627752126547661]

,targetid,z,zerr,zwarn,civ_1549_flux,civ_1549_flux_ivar,civ_1549_sigma,flux_w3,flux_ivar_w3,spectype,survey
237645,39627752126547661,2.514396,0.000122,0,1500.8134,0.005692,2883.1355,457.9774,0.001518,QSO,sv3
237646,39627752126547661,2.514785,0.000058,0,1276.1000,0.031510,2690.0967,457.9774,0.001518,QSO,main
237647,39627752126547661,2.515018,0.000057,0,1175.9384,0.044369,2507.1800,457.9774,0.001518,QSO,sv1
237648,39627752126547661,2.514477,0.000041,0,1133.3832,0.048559,2408.4430,457.9774,0.001518,QSO,sv3
237649,39627752126547661,2.514536,0.000257,0,1416.8228,0.001593,2782.9280,457.9774,0.001518,QSO,main


In [16]:
# path = "data/local/query_results_from_dr1_agnqso.csv"
# # results.to_csv(path)

# results = pd.read_csv(path,index_col=0)
# results.describe()

In [19]:
results["snr_w3"] = results["flux_w3"] * np.sqrt(results["flux_ivar_w3"])
results["civ_1549_fwhm"] = results["civ_1549_sigma"] * 2.355

In [20]:
results.query("snr_w3 > 3 & civ_1549_fwhm > 500 & civ_1549_fwhm < 20000")

,targetid,z,zerr,zwarn,civ_1549_flux,civ_1549_flux_ivar,civ_1549_sigma,flux_w3,flux_ivar_w3,spectype,survey,civ_1549_fwhm,snr_w3
3,39633221415076682,3.538210,0.000222,0,633.759600,0.011682,2062.5470,178.974930,0.003023,QSO,main,4857.298185,9.840349
4,39633221415076682,3.538506,0.000109,0,821.267000,0.028074,2820.0908,178.974930,0.003023,QSO,main,6641.313834,9.840349
6,39633217371767461,2.962281,0.000715,0,129.613590,0.048463,3775.2640,57.191692,0.003005,QSO,main,8890.746720,3.134989
14,39633221402494356,2.574769,0.000631,0,130.015520,0.031509,2685.2673,69.645035,0.003883,QSO,main,6323.804492,4.339632
15,39633221398303680,2.981374,0.000640,0,89.910680,0.066117,2263.1199,84.583810,0.004306,QSO,main,5329.647365,5.550246
...,...,...,...,...,...,...,...,...,...,...,...,...,...
404882,39633154423653200,2.736704,0.000877,0,33.968170,0.220137,217.5175,205.015780,0.002997,QSO,main,512.253713,11.223752
404889,39633158747980629,3.454700,0.000592,0,59.335136,0.121402,1276.2297,86.342606,0.002717,QSO,main,3005.520944,4.500355
404890,39633163047142707,2.211673,0.001106,0,91.779730,0.016726,2806.7650,102.519100,0.002832,QSO,main,6609.931575,5.455618
404892,39633163047144494,2.447516,0.000300,0,668.319030,0.011542,2872.7651,73.134560,0.002893,QSO,main,6765.361811,3.933513


In [19]:
# def adjust_z_lam(wavelength,z):
#     return wavelength/(1+z)
# adjust_z_lam(3600,3.4)

In [8]:
# client = SparclClient()

# idxs = results["targetid"].to_list()

# idxs = [int(x) for x in idxs]

# output_fields = ['specid', 'redshift', 'flux', 'wavelength', 'spectype', 'specprimary','survey', 'data_release', 'targetid', 'redshift_warning', 'desi_target']

In [9]:
# spectra = client.retrieve_by_specid(specid_list=idxs[:100], include=output_fields,dataset_list=["DESI-DR1"])

In [10]:
# spectra_df = pd.DataFrame(spectra.data[1:])
# spectra_df

In [11]:
# idx = 9


# lam = spectra_df["wavelength"][idx]
# flux = spectra_df["flux"][idx]
# z = spectra_df["redshift"][idx]

# lam, flux = adjust_redshift(lam, flux, z)

# plot_spectrum(lam, flux, line_lambda=1548, line_label="CIV")